# Notebook 054: Structured Streaming Foundations: Sources, Execution Model and Query Progress

**Module:** Module 06: Streaming & Near Real-Time Processing  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 11: Auto-loader & Video 17: Lakeflow Declarative Pipelines  

---

## 1. Business Scenario & Real-World Context
A payments platform monitors live transaction throughput. As card swipes occur across the globe, an unbounded stream must be parsed, enriched, and aggregated in real time without dropping micro-batches.

## 2. Technical Requirement & Problem Statement
Master Spark Structured Streaming fundamentals: create a streaming DataFrame from the memory/rate stream source, inspect stream schema, configure streaming queries (`readStream` / `writeStream`), monitor execution metrics via `StreamingQueryProgress`, and manage query lifecycle.

## 3. Spark & Databricks Architecture Notes
Spark Structured Streaming treats a live stream as an unbounded, continuously appended table. The engine processes micro-batches incrementally, maintaining offsets in a write-ahead log (WAL) and state in RocksDB/memory, providing end-to-end exactly-once fault tolerance.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_054_Structured_Streaming_Foundations_Sources,_Execution_Model_and_Query_Progress")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating an Unbounded Synthetic Streaming Source
Use Spark's built-in 'rate' stream source to generate synthetic streaming timestamps.

In [ ]:
# rate source generates timestamp and monotonic value at configured rows per second
df_stream_raw = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numPartitions", 2)
    .load()
)

print(f"Is DataFrame streaming: {df_stream_raw.isStreaming}")
df_stream_raw.printSchema()

## 6. Streaming Transformations: Deriving Live Transaction Metrics
Apply standard DataFrame transformations over the unbounded streaming dataset.

In [ ]:
df_transactions_stream = df_stream_raw.select(
    F.col("timestamp").alias("event_time"),
    F.concat(F.lit("TXN-"), F.col("value")).alias("transaction_id"),
    F.round((F.col("value") % 500) + 10.50, 2).alias("amount"),
    F.when(F.col("value") % 10 == 0, "HIGH_RISK").otherwise("STANDARD").alias("risk_category")
)

df_transactions_stream.printSchema()

## 7. Starting Memory Sink & Inspecting StreamingQueryProgress
Start stream writing to in-memory table for interactive querying and inspect progress metrics.

In [ ]:
import time

# Start streaming query writing to in-memory table
query = (
    df_transactions_stream.writeStream
    .format("memory")
    .queryName("live_transactions")
    .outputMode("append")
    .start()
)

print(f"Streaming query started! ID: {query.id} | Status: {query.status['message']}")

# Let the stream run for 3 seconds to accumulate micro-batches
time.sleep(3)

# Query recent progress
progress = query.lastProgress
if progress:
    print(f"Input Rate: {progress.get('inputRowsPerSecond', 0):.1f} rows/sec")
    print(f"Processing Rate: {progress.get('processedRowsPerSecond', 0):.1f} rows/sec")
    print(f"Batch ID: {progress.get('batchId', 0)}")

# Query the in-memory streaming table using Spark SQL
print("Live In-Memory Streaming Snapshot:")
spark.sql("SELECT * FROM live_transactions ORDER BY event_time DESC LIMIT 5").show(truncate=False)

# Safely stop streaming query
query.stop()
print("Streaming query safely stopped.")

## 9. Key Architectural Takeaways
- **Structured Streaming abstracts streaming into an unbounded table queried using standard DataFrame API operations.**
- **Output modes: `append` (emits new rows), `update` (emits changed rows), `complete` (emits entire aggregated table).**
- **`query.lastProgress` provides vital telemetry: input rows per second, processing latency, and state memory.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between micro-batch execution and Continuous Processing mode in Spark Streaming?
2. How does the checkpoint directory guarantee exactly-once processing during executor node crashes?
3. What are the three output modes in Structured Streaming and when must each be used?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
